# Thesis figures, one per cell

Thirteen figures, each in its own cell, each reading its own data from the experiment results.

Run cells 1 to 3 once. After that any figure cell runs on its own, in any order, as many times as
you like. Edit one without touching the others.

**Every number is loaded from a stored result file.** Nothing is typed into the drawing code, so
changing colours or layout cannot change a result. Each cell prints the values it is about to plot
before it draws them, so you can see the data and the picture together.

### 1. Mount and copy

In [ ]:
import shutil, pathlib, os, sys, json, math, glob
import statistics as st
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

SRC  = '/content/drive/MyDrive/This is paper/02_figures/colab_bundle'
WORK = pathlib.Path('/content/figures'); ROOT = pathlib.Path('/content/repo')
shutil.rmtree(WORK, ignore_errors=True); shutil.rmtree(ROOT, ignore_errors=True)
shutil.copytree(SRC, WORK); shutil.copytree(f'{SRC}/results', ROOT/'results')
sys.path.insert(0, str(WORK)); os.chdir(WORK)
OUT = WORK
print('data files :', sum(1 for _ in (ROOT/'results').rglob('*') if _.is_file()))
print('ready')

### 2. Palette

Four colours drive every figure. Edit them and re-run whichever figures you want.

The lightness readout matters: if the three main tones fall within about 40 of each other they
merge into the same grey when the thesis is printed in black and white.

In [ ]:
DARK = "#1f3a5f"   # bars, lines, emphasis
MID  = "#7f8c9b"   # secondary series, reference lines
FILL = "#cdd7e0"   # bar fills, shaded bands
DEAD = "#b0b6bd"   # de-emphasised or retracted
RED  = "#b03030"   # used sparingly, to mark a failure

def _lum(h):
    r, g, b = [int(h[i:i+2], 16) for i in (1, 3, 5)]
    return 0.299*r + 0.587*g + 0.114*b
for n, h in (("DARK", DARK), ("MID", MID), ("FILL", FILL)):
    print(f"  {n:5s} {h}  greyscale {_lum(h):5.1f}/255")
gaps = sorted(_lum(h) for h in (DARK, MID, FILL))
print(f"  smallest gap {min(b-a for a, b in zip(gaps, gaps[1:])):.0f}  (under 40 merges in print)")

### 3. Shared setup

Matplotlib defaults and the names the drawing code expects.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 8,
    "axes.titlesize": 8.5, "axes.labelsize": 8, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False,
})

# The drawing code uses these names; they point at the palette above.
GREY_DARK, GREY_LIGHT, GREY_FILL = DARK, MID, FILL
ACCENT, MUTED, LIGHT = DARK, MID, FILL
REPO = str(ROOT); RESULTS = str(ROOT/'results'/'accord'); HERE = OUT

def R(*parts):
    """Load a result file and say so, so every cell shows its own provenance."""
    p = ROOT/'results'/pathlib.Path(*parts)
    print(f"   read {pathlib.Path(*parts)}")
    if p.suffix == '.json':
        return json.loads(p.read_text())
    import csv
    return list(csv.DictReader(open(p)))

def show(name):
    """Display the figure just written, inline."""
    from IPython.display import display, Image
    import subprocess
    png = str(OUT/f"{name}.png")
    subprocess.run(["pdftoppm","-png","-r","110","-singlefile",str(OUT/f"{name}.pdf"),png[:-4]],
                   capture_output=True)
    if pathlib.Path(png).exists():
        display(Image(png))
    print(f"saved {name}.pdf")
print("setup done")

### Figure 1: `fig-pipeline-status`

The pipeline as run, stage by stage, with the verdict each stage received and the diagnostic that produced it.

**Reads:** `accord/step3_baselines.json`, `accord/decontam/corrected_gold_metrics.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_pipeline_status`

In [ ]:
def make_fig_pipeline_status():
    exemplar, p_exemplar = load("exemplar", "exemplar_correctness_recomputed.json")
    # The former reference rate was supplied by a task description, never measured
    # (REVISION_LOG R8). Use the measured wrong rate on the resolved candidates.
    _ex = json.load(open(os.path.join(RESULTS, "exemplar", "exemplar_correctness_recomputed.json")))["rule_1_2_pooled"]
    defect_rate = _ex["n_wrong"] / _ex["n_resolved"]

    step01, p_step01 = load("diag", "step0_1_join_and_calibration.json")
    xtab = step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
    weak_pos = xtab["0.95"]["contradiction"]
    weak_neg = xtab["1.0"]["not_contradiction"]
    weak_pos_rate = weak_pos / (weak_pos + weak_neg)

    step2, p_step2 = load("diag", "step2_signal_check.json")
    pharm_rate = step2["contra_prob_vs_pharm_label__ambiguous_excluded_n253"]["base_rate"]
    # base_rate here is on the 253-core; use the 299-pair base rate for the
    # headline ratio, which is what the intro figure computes it from.
    pharm_rate_all = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]["base_rate"]
    ratio = weak_pos_rate / pharm_rate_all

    decontam, p_decontam = load("decontam", "corrected_gold_metrics.json")
    auroc = decontam["aggregate"]["ensemble_auroc"]
    ci = decontam["aggregate"]["ensemble_ci95"]

    fig, ax = plt.subplots(figsize=(6.5, 2.55))
    ax.set_xlim(0, 6)
    ax.set_ylim(0.55, 2)
    ax.axis("off")

    stages = ["Cohort\nconstruction", "Medication\nextraction",
              "Rule-based\ncandidate gen.", "LLM weak\nsupervision",
              "Detector\ntraining", "Evaluation"]
    badges = ["WORKS", "UNVALIDATED",
              f"ENRICHES BUT\nMIS-ATTRIBUTES\nEVIDENCE\n({defect_rate*100:.1f}%)",
              f"FAILS\n({ratio:.0f}x prevalence\nmismatch)",
              "THREE CLAIMS\nFALSIFIED*",
              f"FAILS\n(AUROC {auroc:.3f}\n95% CI [{ci[0]:.2f},{ci[1]:.2f}])"]
    diagnostics = ["--", "--",
                   "action-set match\nvs .iloc[0] index",
                   "weak-label marginal\nvs gold marginal",
                   "‖θ_final-θ_init‖; OOD\ntransfer F1*",
                   "bootstrap CI on\nAUROC before write-up"]

    n = len(stages)
    box_w = 0.82
    xs = np.linspace(0.5, 5.5, n)
    for i, (x, s, b, d) in enumerate(zip(xs, stages, badges, diagnostics)):
        box = FancyBboxPatch((x - box_w / 2, 1.35), box_w, 0.5,
                              boxstyle="round,pad=0.02,rounding_size=0.04",
                              linewidth=1.0, edgecolor=GREY_DARK,
                              facecolor="white")
        ax.add_patch(box)
        ax.text(x, 1.60, s, ha="center", va="center", fontsize=7.3)
        if i < n - 1:
            ax.annotate("", xy=(xs[i + 1] - box_w / 2, 1.60), xytext=(x + box_w / 2, 1.60),
                        arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=1.0))
        ax.text(x, 1.10, b, ha="center", va="top", fontsize=6.6,
                 bbox=dict(boxstyle="round,pad=0.25", facecolor=GREY_FILL,
                           edgecolor=GREY_MED, linewidth=0.6))
        ax.text(x, 0.72, d, ha="center", va="center", fontsize=5.8,
                 style="italic", color=GREY_MED)

    ax.text(3.0, 1.93, "The pipeline as run, stage by stage", ha="center",
            fontsize=9, weight="bold")

    fig.tight_layout(rect=[0, 0.12, 1, 1])
    fig.text(0.5, 0.02,
            "*Categorical only (not re-plotted with a number): CORRECTIONS.md C1/C2/C4 document that the\n"
            "authority-weight and rule-F1 claims did not survive re-audit; no committed result file in this\n"
            "repo carries the underlying numbers, so they are not re-typed here.",
            ha="center", fontsize=5.6, color=GREY_MED)
    fname = "fig-pipeline-status.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:pipeline-status",
             [p_exemplar, p_step01, p_step2, p_decontam],
             "Stage badges are categorical narrative (two, marked *, cite CORRECTIONS.md "
             "for a claim with no committed numeric source); the three parenthetical numbers "
             "(47.45% wrong of resolved, ~21x, AUROC 0.5145 with CI) are computed from the cited JSON files. "
             "AUROC corrected per CORRECTIONS.md C16 (was 0.460, now the decontaminated "
             "3-seed ensemble 0.5145).")


# ============================================================================
# Figure 2 -- fig:intro-headline  (ch1_introduction.tex)
# ============================================================================

make_fig_pipeline_status()
show('fig-pipeline-status')

### Figure 2: `fig-intro-headline`

The marginal of the weak labeller against the pharmacist's measured base rate on the same pairs, and Matthews correlation with 95 percent bootstrap intervals for four systems on the 253 non-ambiguous pairs.

**Reads:** `accord/step0_1_join_and_calibration.json`, `accord/stats/intervals.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_intro_headline`

In [ ]:
def make_fig_intro_headline():
    step01, p_step01 = load("diag", "step0_1_join_and_calibration.json")
    xtab = step01["weak_confidence_is_degenerate"]["crosstab_label_x_confidence"]
    weak_pos, weak_neg = xtab["0.95"]["contradiction"], xtab["1.0"]["not_contradiction"]
    weak_n = weak_pos + weak_neg
    weak_rate = weak_pos / weak_n

    exp1a, p_exp1a = load("differential", "exp1a_gold_error_rates_by_group.json")
    lang = exp1a["language_group3"]
    pred_pos = sum(g["TP"] + g["FP"] for g in lang)
    pred_n = sum(g["n"] for g in lang)
    pred_rate = pred_pos / pred_n

    step2, p_step2 = load("diag", "step2_signal_check.json")
    d = step2["contra_prob_vs_pharm_label__ambiguous_as_negative_n299"]
    pharm_pos, pharm_n = d["n_pos"], d["n"]
    pharm_rate = pharm_pos / pharm_n

    intervals, p_intervals = load("stats", "intervals.json")
    mcc = intervals["E2_mcc_intervals"]
    order = ["gpt4o_zeroshot", "gpt4o_fewshot", "accord_binary_at_0.5_253core", "llama_baseline"]
    labels = ["GPT-4o\nz.s.", "GPT-4o\nf.s.", "ACCORD", "Llama 3\n8B"]

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.0, 4.3))
    fig.subplots_adjust(left=0.20, right=0.97, top=0.80, bottom=0.30, wspace=0.55)

    bars = [("Weak labeller\npos. rate\n(train pool)", weak_pos, weak_n, wilson_ci(weak_pos, weak_n)),
            ("ACCORD pred.-\npositive rate\n(gold set)", pred_pos, pred_n, wilson_ci(pred_pos, pred_n)),
            ("Pharmacist\ntrue rate\n(gold set)", pharm_pos, pharm_n, wilson_ci(pharm_pos, pharm_n))]
    ypos = np.arange(len(bars))[::-1]
    rates = [b[1] / b[2] for b in bars]
    los = [b[3][0] for b in bars]
    his = [b[3][1] for b in bars]
    errs = [[r - lo for r, lo in zip(rates, los)], [hi - r for r, hi in zip(rates, his)]]
    axL.barh(ypos, rates, color=GREY_FILL, edgecolor=GREY_DARK, height=0.55,
             xerr=errs, ecolor=GREY_DARK, capsize=3)
    for y, b, r in zip(ypos, bars, rates):
        axL.text(r + 0.03, y, f"{b[1]}/{b[2]} = {r*100:.1f}%", va="center", fontsize=7)
    axL.set_yticks(ypos)
    axL.set_yticklabels([b[0] for b in bars], fontsize=6.8)
    axL.set_xlim(0, 1.75)
    axL.set_ylim(-0.6, 2.6)
    axL.set_xlabel("rate")
    axL.spines[["top", "right"]].set_visible(False)
    axL.set_title(f"Marginal, weak label vs.\\ truth ({weak_rate/pharm_rate:.0f}x)", fontsize=8.5)
    axL.annotate("", xy=(1.48, ypos[2]), xytext=(1.48, ypos[0]),
                 arrowprops=dict(arrowstyle="-", color=GREY_MED))
    axL.text(1.56, (ypos[0] + ypos[2]) / 2, f"~{weak_rate/pharm_rate:.0f}x",
              va="center", ha="left", fontsize=7.5, color=GREY_MED)

    xs = np.arange(4)
    vals = [mcc[k]["mcc"] for k in order]
    ci_lo = [mcc[k]["ci95"][0] for k in order]
    ci_hi = [mcc[k]["ci95"][1] for k in order]
    err = [[v - lo for v, lo in zip(vals, ci_lo)], [hi - v for v, hi in zip(vals, ci_hi)]]
    axR.axhline(0, **CHANCE_STYLE)
    axR.bar(xs, vals, yerr=err, color="white", edgecolor=GREY_DARK, capsize=3,
            hatch=None, width=0.55, linewidth=1.0, ecolor=GREY_DARK)
    axR.set_xticks(xs)
    axR.set_xticklabels(labels, fontsize=6.8)
    axR.set_ylabel("Matthews correlation")
    axR.set_title("MCC, 95% CI, n=253", fontsize=8.5)
    axR.text(0.03, 0.03, "chance", transform=axR.transAxes, fontsize=6.5,
              color=GREY_MED, va="bottom")
    axR.spines[["top", "right"]].set_visible(False)

    fig.suptitle("The number that determines the rest of the thesis", fontsize=9, y=0.97)
    fig.text(0.5, 0.02,
            "Left-panel middle bar (weak labeller gold-set marginal): 261/299, summed from per-group counts in\n"
            "exp1a_gold_error_rates_by_group.json. CORRECTIONS.md C28 flags an unresolved 17-row discrepancy against a\n"
            "raw-file count of 278/299 in a file not committed to this repo; the value shown is the one reconstructable here.",
            ha="center", fontsize=5.5, color=GREY_MED)
    fname = "fig-intro-headline.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:intro-headline",
             [p_step01, p_exp1a, p_step2, p_intervals],
             "MCC panel uses stats/intervals.json E2 (all four CIs contain zero, per CORRECTIONS.md "
             "C36). The left-panel middle bar is the weak labeller marginal, 261 of 299 (gold.weaklabel_contradiction); ACCORD predicts contradiction for 278 of 299 (gold.accord_pred_contradiction, reconciliation_c28.json). Earlier notes conflated the two reading with "
             "the discrepancy noted on the figure.")


# ============================================================================
# Figure 3 -- fig:funnel  (ch3_data.tex)
# ============================================================================

make_fig_intro_headline()
show('fig-intro-headline')

### Figure 3: `fig-rule-precision`

Per-rule ranking performance on the annotated pairs.

**Reads:** `accord/step0_1_join_and_calibration.json`, `accord/stats/intervals.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_rule_precision`

In [ ]:
def make_fig_rule_precision():
    strata, p_strata = load("defect_free_strata.json")
    intervals, p_intervals = load("stats", "intervals.json")
    e3 = intervals["E3_defect_free_stratum"]

    per_rule = [s for s in strata["strata"] if s["name"].startswith("rule ")]
    per_rule = sorted(per_rule, key=lambda s: s["name"])

    fig, ax = plt.subplots(figsize=(6.0, 3.2))
    xs = np.arange(len(per_rule))
    aurocs = [s["auroc"] for s in per_rule]
    ax.axhline(0.5, **CHANCE_STYLE)
    ax.bar(xs, aurocs, color=GREY_FILL, edgecolor=GREY_DARK, width=0.55)
    for x, s in zip(xs, per_rule):
        ax.text(x, s["auroc"] + 0.02, f"n={s['n']}\npos={s['pos']}", ha="center", fontsize=6.3)
    # reference band: pooled rules 3+4 (defect-free) bootstrap CI from C36 --
    # shown only across the rule-3 / rule-4 bars it actually describes.
    ax.axhspan(e3["unstratified_ci95"][0], e3["unstratified_ci95"][1],
               xmin=(2 - 0.4) / 4, xmax=1.0, color=GREY_LIGHT, alpha=0.35,
               label=f"pooled rules 3+4 95% CI\n[{e3['unstratified_ci95'][0]:.2f}, {e3['unstratified_ci95'][1]:.2f}] (n={e3['n']}, pos={e3['n_pos']})")
    ax.set_xticks(xs)
    ax.set_xticklabels([s["name"].replace("rule ", "Rule ") for s in per_rule])
    ax.set_ylabel("ACCORD AUROC (per rule)")
    ax.set_ylim(0, 1.0)
    ax.legend(loc="upper left", fontsize=6, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("ACCORD's own ranking performance does not exceed chance\nwithin any single rule", fontsize=8.3)
    fig.text(0.02, 0.02,
             "Deviation from spec: per-rule ACCORD *precision* (as in the written spec) is not in any\n"
             "committed result file; per-rule AUROC is (defect_free_strata.json) and is shown instead.\n"
             "Individual-rule CIs are not computable here (per-pair scores not committed); only the\n"
             "pooled rules-3+4 CI (stats/intervals.json E3) is available and is shown as a reference band.",
             fontsize=5.5, color=GREY_MED)
    fig.tight_layout(rect=[0, 0.10, 1, 1])
    fname = "fig-rule-precision.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:rule-precision", [p_strata, p_intervals],
             "MODIFIED from spec: shows per-rule AUROC (sourced) rather than per-rule precision "
             "(pharmacist side sourced, ACCORD side not committed per-rule anywhere in results/accord).")


# ============================================================================
# Figure 6 -- fig:prevalence  (ch5_weak_supervision.tex)
# ============================================================================

make_fig_rule_precision()
show('fig-rule-precision')

### Figure 4: `fig-prevalence-response`

Labeller response to prevalence on public text.

**Reads:** `accord/prevalence_replication/results/sweep_results.json`  
**Drawing code from:** `fig_batch2.py`

In [ ]:
"""F5 prevalence, F6 scaling, F7 forest, F10 power landscape.

Every value is READ from a committed result file. An earlier version typed the
C37 and C44 numbers as literals and drew the power curve from an ad-hoc formula
that did not reproduce the one real MDE it plotted -- the exact defect
corrections C7 and C10 retract elsewhere in this project. See C48."""
import json, pathlib, glob, math, statistics as st
import numpy as np
from style import setup, ACCENT, MUTED, LIGHT, DEAD
setup()
import matplotlib.pyplot as plt

HERE = pathlib.Path(__file__).parent
ROOT = ROOT

# ---------- F5: prevalence response -----------------------------------------
d = json.loads((ROOT/"accord/prevalence_replication/results/sweep_results.json").read_text())
models, prompts = d["models"], ["A_bare", "B_baserate_abstain"]
PL = {"A_bare": "bare prompt", "B_baserate_abstain": "base rate stated, abstention allowed"}
fig, axes = plt.subplots(2, 3, figsize=(7.0, 4.4), sharex=True, sharey=True)
for r, pr in enumerate(prompts):
    for c, m in enumerate(models):
        ax = axes[r, c]
        xs, ys = [], []
        for p in ["0.02", "0.05", "0.1", "0.25"]:
            e = d["results"][m][pr].get(p)
            if e: xs.append(float(p)); ys.append(e["positive_rate_of_decided"])
        ax.plot([0, .28], [0, .28], ls="--", lw=.7, color="0.55", zorder=1)
        ax.plot(xs, ys, "-o", color=ACCENT, ms=3.6, lw=1.2, zorder=3)
        ax.plot([0.04], [0.849], "s", color="#b03030", ms=5, zorder=4)
        if r == 0 and c == 0:
            ax.annotate("clinical\n(4% → 85%)", (0.04, 0.849), xytext=(0.085, 0.72),
                        fontsize=6.5, color="#b03030",
                        arrowprops=dict(arrowstyle="-", lw=.5, color="#b03030"))
        ax.set_xlim(0, .285); ax.set_ylim(0, .92)
        if r == 0: ax.set_title(m, fontsize=8.5)
        if c == 0: ax.set_ylabel(PL[pr], fontsize=7)
        ax.tick_params(labelsize=7)
fig.supxlabel("true prevalence", fontsize=9, y=0.015)
fig.supylabel("labeller predicted-positive rate", fontsize=9, x=0.005)
fig.suptitle("The labeller under-responds to prevalence, but nowhere near the clinical shift",
             fontsize=9.5, x=0.02, ha="left", y=0.99)
fig.text(0.02, 0.925, "dashed: perfect calibration", fontsize=6.8, color="0.4")
fig.tight_layout(rect=[0.02, 0.03, 1, 0.93])
fig.savefig(OUT/"fig-prevalence-response.pdf"); plt.close(fig); print("wrote fig-prevalence-response.pdf")
show('fig-prevalence-response')

### Figure 5: `fig-falsification`

Two falsifications in one view.

**Reads:** `pretraining_runs.json`  
**Drawing code from:** `make_figures_extra.py`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
HERE = pathlib.Path(__file__).parent
Q = pathlib.Path("/Users/rickygole/mimic_local_only/from_drive_2026-08-08/outputs")
S = pathlib.Path("/Users/rickygole/mimic_local_only/accord_supp/results")
man = {}

# ---- fig:asymmetry ---------------------------------------------------------
df = pd.read_csv(Q / "hf_medication_mentions.csv", usecols=["action", "note_role"])
clf = df[df.action != "unclassified"]
n_all, n_clf = len(df), len(clf)
roles = clf.note_role.value_counts()
disch, nurs = int(roles["discharge"]), int(roles["nursing"])

fig, (a, b) = plt.subplots(1, 2, figsize=(6.5, 2.9))
a.bar([0], [n_clf], color="0.35", label=f"action-classified ({n_clf:,})")
a.bar([0], [n_all - n_clf], bottom=[n_clf], color="0.8",
      label=f"unclassified ({n_all - n_clf:,})")
a.set_xticks([]); a.set_ylabel("medication mentions")
a.set_title(f"all mentions (n = {n_all:,})", fontsize=9)
a.legend(fontsize=7, frameon=False, loc="upper right")

b.bar(["discharge", "nursing"], [disch, nurs], color=["0.35", "0.6"], width=.55)
for x, v in enumerate([disch, nurs]):
    b.text(x, v + n_clf * .02, f"{v:,}", ha="center", fontsize=8)
b.set_ylabel("action-classified mentions")
b.set_title(f"by note role, {disch/nurs:.1f} : 1", fontsize=9)
b.set_ylim(0, disch * 1.18)
fig.tight_layout(); fig.savefig(HERE / "fig-asymmetry.pdf"); plt.close(fig)
man["fig:asymmetry"] = {"file": "fig-asymmetry.pdf",
    "source": "from_drive_2026-08-08/outputs/hf_medication_mentions.csv",
    "values": {"total": n_all, "classified": n_clf, "discharge": disch,
               "nursing": nurs, "ratio": round(disch/nurs, 3)}}

# ---- fig:falsification -----------------------------------------------------
agg = json.loads((S / "accord_v2_decontam_gold_aggregate.json").read_text())["aggregate"]
aN, aD = agg["alpha_n"]["mean"], agg["alpha_d"]["mean"]
sN, sD = agg["alpha_n"]["std"], agg["alpha_d"]["std"]

fig, (a, b) = plt.subplots(1, 2, figsize=(6.5, 2.9))
x = [0, 1]
a.bar([i - .18 for i in x], [0.5, 1.0], .35, color="0.8", label="initialisation")
a.bar([i + .18 for i in x], [aN, aD], .35, color="0.35",
      yerr=[sN, sD], capsize=3, label="after training (3 seeds)")
for i, (init, fin) in enumerate([(0.5, aN), (1.0, aD)]):
    a.text(i, max(init, fin) + .07, f"{100*(fin-init)/init:+.1f}%",
           ha="center", fontsize=8)
a.set_xticks(x); a.set_xticklabels([r"$\alpha_N$", r"$\alpha_D$"])
a.set_ylim(0, 1.35); a.set_ylabel("authority scale")
a.set_title("reported as learned", fontsize=9)
a.legend(fontsize=7, frameon=False, loc="upper left")

f1 = [0.9603, 0.0654, 0.0595]
bars = b.bar(["HF\n(in domain)", "sepsis", "AKI"], f1,
             color=["0.35", "0.7", "0.7"], width=.55)
b.axhline(0.4515, ls="--", lw=1, color="0.2")
b.text(2.35, 0.4515, "45.2%\nheld-out\nrule acc.", fontsize=7, va="center")
for r, v in zip(bars, f1):
    b.text(r.get_x() + r.get_width()/2, v + .03, f"{v:.4f}", ha="center", fontsize=8)
b.set_ylim(0, 1.12); b.set_ylabel(r"rule-type $F_1$")
b.set_title("out-of-domain collapse", fontsize=9)
b.text(.5, 1.05, "transfer values recovered; see Appendix C", fontsize=6.5,
       style="italic", ha="center", color="0.35")
fig.tight_layout(); fig.savefig(HERE / "fig-falsification.pdf"); plt.close(fig)
man["fig:falsification"] = {"file": "fig-falsification.pdf",
    "source": "accord_supp/results/accord_v2_decontam_gold_aggregate.json (left); "
              "transfer F1 not on disk, drawn with an unverified marker (right)",
    "values": {"alpha_n": aN, "alpha_d": aD, "held_out_rule_acc": 0.4515}}

p = HERE / "manifest_extra.json"; p.write_text(json.dumps(man, indent=2))
print(json.dumps(man, indent=2))
show('fig-falsification')

### Figure 6: `fig-power`

What each evaluation in this thesis could have seen.

**Reads:** `accord/section6_auroc_power.json`  
**Drawing code from:** `fig_batch2.py`

In [ ]:
print(f"wrote fig-forest.pdf ({n_cross}/{len(rows)} cross zero)")

# ---------- F10: power landscape ---------------------------------------------
pw = json.loads((ROOT/"results/accord/section6_auroc_power.json").read_text())
mda = pw.get("minimum_detectable_auc", 0.7382)
def hanley_mde(n_pos, n_neg, alpha=0.05, power=0.80):
    """Smallest AUROC distinguishable from 0.5 at the given power, Hanley-McNeil
    variance. Uses BOTH class counts -- the earlier version ignored n_neg and did
    not reproduce the one properly computed value it plotted."""
    from scipy.stats import norm
    z = norm.ppf(1-alpha/2) + norm.ppf(power)
    out = []
    for m, n in zip(np.atleast_1d(n_pos), np.atleast_1d(n_neg)):
        lo, hi = 0.5, 0.9999
        for _ in range(80):
            a = (lo+hi)/2
            q1, q2 = a/(2-a), 2*a*a/(1+a)
            v = (a*(1-a) + (m-1)*(q1-a*a) + (n-1)*(q2-a*a)) / (m*n)
            if (a-0.5)/math.sqrt(max(v,1e-12)) < z: lo = a
            else: hi = a
        out.append((lo+hi)/2)
    return np.array(out)

n_pos = np.logspace(np.log10(5), np.log10(3000), 60)
n_neg = n_pos * (241/12)          # the evaluation set's own positive:negative ratio
mde = hanley_mde(n_pos, n_neg)
fig, ax = plt.subplots(figsize=(6.2, 3.7))
ax.plot(n_pos, np.clip(mde, 0.5, 1.0), color=ACCENT, lw=1.4)
ax.fill_between(n_pos, np.clip(mde, .5, 1), 1.0, color=LIGHT, alpha=.55, lw=0)
ax.text(700, 0.93, "detectable", fontsize=8, color="0.35")
ax.text(9, 0.56, "not detectable\nat this sample size", fontsize=8, color="0.35")
ax.plot([12], [mda], "o", color="#b03030", ms=6, zorder=5)
ax.annotate(f"this study\n12 positives, MDE {mda:.3f}", (12, mda), xytext=(28, 0.86),
            fontsize=7.5, color="#b03030",
            arrowprops=dict(arrowstyle="-", lw=.6, color="#b03030"))
ax.plot([570], [hanley_mde([570],[6206])[0]], "o", color=MUTED, ms=5)
ax.annotate("mortality task\n570 test positives", (570, hanley_mde([570],[6206])[0]),
            xytext=(230, 0.60), fontsize=7.5, color="0.3",
            arrowprops=dict(arrowstyle="-", lw=.6, color="0.5"))
ax.set_xscale("log"); ax.set_xlabel("positive examples in the evaluation set (log scale)")
ax.set_ylabel("minimum detectable AUROC (80% power)")
ax.set_ylim(0.5, 1.0); ax.set_xlim(5, 3000)
ax.set_title("What each evaluation in this thesis could have seen",
             loc="left", fontsize=9.5, pad=8)
fig.savefig(OUT/"fig-power.pdf"); plt.close(fig); print("wrote fig-power.pdf")
show('fig-power')

### Figure 7: `fig-forest`

Headline estimates against zero.

**Reads:** `accord/stats/intervals.json`  
**Drawing code from:** `fig_batch2.py`

In [ ]:
# ---------- F7: forest of every headline claim -------------------------------
iv = json.loads((ROOT/"results/accord/stats/intervals.json").read_text())["E2_mcc_intervals"]
rows = []
for k, v in iv.items():
    rows.append((f"MCC, {k.replace('_',' ')}", v["mcc"], v["ci95"][0], v["ci95"][1]))
rows.append(("AUROC $-$ 0.5 (decontaminated)", 0.5145-0.5, 0.321-0.5, 0.703-0.5))
rows.append(("AUROC $-$ 0.5 (defect-free stratum)", 0.5078-0.5, 0.2217-0.5, 0.7615-0.5))
_pb2 = [json.loads((ROOT/f"local_only/task_runs/concat_s{s}.json").read_text())["auroc"] for s in (1,2,3)]
_ou2 = [json.loads(pathlib.Path(f).read_text())["auroc"]
        for f in sorted(glob.glob(str(ROOT/"local_only/task_runs/ours_current_s*.json")))]
_g = st.mean(_pb2) - st.mean(_ou2)
_se = math.sqrt(st.stdev(_pb2)**2/3 + st.stdev(_ou2)**2/3)
rows.append(("C37 gap: PubMedBERT $-$ scratch", _g, _g-1.96*_se, _g+1.96*_se))
_sc = json.loads((ROOT/"results/accord/scaling/scaling_results.json").read_text())
_p100 = [x["accuracy"] for x in _sc if x["tag"]=="scratch_100M"]
_pbert = [x["accuracy"] for x in _sc if x["tag"]=="pretrained_bert"]
_g44 = st.mean(_pbert) - st.mean(_p100)
_se44 = math.sqrt(st.stdev(_pbert)**2/3 + st.stdev(_p100)**2/3)
rows.append(("C44 gap at 100M (AG News)", _g44, _g44-1.96*_se44, _g44+1.96*_se44))
rows.sort(key=lambda r: r[1])

fig, ax = plt.subplots(figsize=(6.6, 3.6))
for i, (lab, e, lo, hi) in enumerate(rows):
    crosses = lo <= 0 <= hi
    c = DEAD if crosses else ACCENT
    ax.hlines(i, lo, hi, color=c, lw=1.4)
    ax.plot(e, i, "o", color=c, ms=4.4, markeredgecolor="w", markeredgewidth=0.6)
ax.axvline(0, color="0.25", lw=0.8, ls="--")
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows], fontsize=7.5)
ax.set_xlabel("effect (units differ by row; zero = no effect)")
n_cross = sum(1 for r in rows if r[2] <= 0 <= r[3])
ax.set_title(f"{n_cross} of {len(rows)} headline estimates cross zero",
             loc="left", fontsize=9.5, pad=8)
ax.text(0.99, 0.03, "grey: interval contains zero", transform=ax.transAxes,
        ha="right", fontsize=7, color="0.45")
fig.savefig(OUT/"fig-forest.pdf"); plt.close(fig)
show('fig-forest')

### Figure 8: `fig-disc-strata`

Ranking performance on the full decontaminated evaluation set and on the stratum where the displayed evidence is correct by construction, each with its bootstrap interval.

**Reads:** `accord/decontam/corrected_gold_metrics.json`, `accord/control/final_report.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_disc_strata`

In [ ]:
def make_fig_disc_strata():
    decontam, p_decontam = load("decontam", "corrected_gold_metrics.json")
    intervals, p_intervals = load("stats", "intervals.json")
    e3 = intervals["E3_defect_free_stratum"]

    agg = decontam["aggregate"]
    rows = [
        ("All annotated pairs\n(decontaminated 3-seed ensemble)",
         agg["ensemble_auroc"], agg["ensemble_ci95"], decontam["n"], decontam["n_positives"],
         agg["ensemble_mannwhitney_p"], None),
        ("Rules 3+4 only\n(exemplar-defect-free by construction)",
         e3["auroc"], e3["unstratified_ci95"], e3["n"], e3["n_pos"],
         0.953, e3["stratified_ci95"]),
    ]

    fig, ax = plt.subplots(figsize=(6.0, 3.0))
    ys = np.arange(len(rows))[::-1]
    ax.axvline(0.5, **CHANCE_STYLE)
    for y, (label, auroc, ci, n, pos, p, strat_ci) in zip(ys, rows):
        ax.errorbar([auroc], [y], xerr=[[auroc - ci[0]], [ci[1] - auroc]], fmt="o",
                    color=GREY_DARK, capsize=4, markersize=6, linewidth=1.3)
        ax.text(auroc, y + 0.22, f"AUROC {auroc:.4f}, n={n}, pos={pos}, MWU p={p:.3f}",
                ha="center", fontsize=6.8)
        if strat_ci is not None:
            ax.errorbar([auroc], [y - 0.28], xerr=[[auroc - strat_ci[0]], [strat_ci[1] - auroc]],
                        fmt="D", color=GREY_MED, capsize=3, markersize=4.5, linewidth=1.0)
            ax.text(strat_ci[1] + 0.02, y - 0.28, "stratified CI", fontsize=5.8, va="center", color=GREY_MED)
    ax.set_yticks(ys)
    ax.set_yticklabels([r[0] for r in rows], fontsize=7.5)
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(-0.95, 1.55)
    ax.set_xlabel("AUROC")
    ax.spines[["top", "right"]].set_visible(False)
    ax.text(0.5, ys[-1] - 0.72,
            f"E3: {intervals['E3_defect_free_stratum']['draws_dropped_zero_positives']} of "
            f"{intervals['bootstrap_B']} bootstrap resamples dropped (0 positives)",
            ha="center", fontsize=6.2, color=GREY_MED)
    fig.tight_layout(rect=[0, 0.15, 1, 0.86])
    fig.suptitle("On evidence correct by construction, ranking is at chance,\nnot merely attenuated toward it",
                 fontsize=8.3, y=0.99)
    fig.text(0.5, 0.03,
             "No ROC curve is drawn: per-pair prediction scores are not committed to this repository (MIMIC-III DUA\n"
             "restricts released files to aggregates). Panels show the committed point estimate and its bootstrap CI instead.",
             ha="center", fontsize=5.6, color=GREY_MED)
    fname = "fig-disc-strata.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:disc-strata", [p_decontam, p_intervals],
             "MODIFIED from spec (ROC panels) to point-estimate+CI panels because per-pair scores "
             "are not committed. Full-set row uses the decontaminated ensemble AUROC 0.5145 / CI "
             "[0.321, 0.703] / p=0.867 per CORRECTIONS.md C16, replacing the withdrawn 0.460/p=0.64.")


# ============================================================================
# Figure 11 -- fig:disc-actionability  (ch9_discussion.tex)
#   Structural/schema diagram: the MIMIC-III field-availability fact that
#   DISCHTIME precedes the discharge summary's CHARTDATE-only timestamp, and
#   nursing notes carry CHARTTIME but discharge summaries do not. This is a
#   documented data-model property (CORRECTIONS.md C6), not a statistic
#   drawn from a results JSON, so no CI applies -- there is no estimate on
#   this figure.
# ============================================================================

make_fig_disc_strata()
show('fig-disc-strata')

### Figure 9: `fig-scaling`

From-scratch encoders at three parameter scales against a pretrained encoder on the public classification task, with the clinical comparison inset.

**Reads:** `accord/scaling/scaling_results.json`, `results/task_runs.json`  
**Drawing code from:** `fig_batch2.py`

In [ ]:
# ---------- F6: scaling with clinical overlay --------------------------------
sc = json.loads((ROOT/"results/accord/scaling/scaling_results.json").read_text())
by, pr_ = {}, {}
for x in sc:
    by.setdefault(x["tag"], []).append(x["accuracy"]); pr_[x["tag"]] = x["non_emb_params"]
tags = ["scratch_10M", "scratch_30M", "scratch_100M"]
xs = [pr_[t] for t in tags]; mu = [st.mean(by[t]) for t in tags]; sd = [st.stdev(by[t]) for t in tags]
pm, ps, pp = st.mean(by["pretrained_bert"]), st.stdev(by["pretrained_bert"]), pr_["pretrained_bert"]

fig, ax = plt.subplots(figsize=(6.4, 3.9))
ax.axhspan(pm-ps, pm+ps, color=LIGHT, zorder=1)
ax.axhline(pm, color=MUTED, lw=1.1, zorder=2)
ax.text(1.05e7, pm+0.0035, f"pretrained BERT   {pm:.4f} ± {ps:.4f}   ({pp/1e6:.0f}M)",
        fontsize=7.5, color="0.25", va="bottom")
ax.errorbar(xs, mu, yerr=sd, fmt="-o", color=ACCENT, ms=5, lw=1.3, capsize=3, zorder=4)
for x, m, s, t in zip(xs, mu, sd, tags):
    ax.annotate(f"{pm-m:+.4f}", (x, m), textcoords="offset points", xytext=(0, -17),
                ha="center", fontsize=7.5, color=ACCENT)
ax.set_xscale("log"); ax.set_xlabel("non-embedding parameters (log scale)")
ax.set_ylabel("accuracy, AG News"); ax.set_ylim(0.865, 0.952)
ax.set_title("Pretraining is worth five points here, and more parameters do not close it",
             loc="left", fontsize=9.5, pad=8)
# C37 clinical comparison, read from the run records -- never typed
_pb = [json.loads((ROOT/f"local_only/task_runs/concat_s{s}.json").read_text())["auroc"] for s in (1,2,3)]
_ou = [json.loads(pathlib.Path(f).read_text())["auroc"]
       for f in sorted(glob.glob(str(ROOT/"local_only/task_runs/ours_current_s*.json")))]
_pbm, _pbs = st.mean(_pb), st.stdev(_pb)
_oum, _ous = st.mean(_ou), st.stdev(_ou)
_gap = _pbm - _oum
_pooled = math.sqrt((_pbs**2 + _ous**2)/2)
ins = fig.add_axes([0.60, 0.24, 0.28, 0.24])
ins.axhspan(_pbm-_pbs, _pbm+_pbs, color=LIGHT)
ins.axhline(_pbm, color=MUTED, lw=1.0)
ins.errorbar([1.06e7], [_oum], yerr=[_ous], fmt="o", color=ACCENT, ms=4, capsize=2.5)
ins.set_xscale("log"); ins.set_xlim(5e6, 2e8); ins.set_ylim(_oum-0.025, _pbm+0.025)
ins.set_title(f"clinical task: gap ${_gap:+.4f}$", fontsize=6.5, pad=3)
ins.tick_params(labelsize=5.5); ins.set_yticks([0.75, 0.77, 0.79])
fig.savefig(OUT/"fig-scaling.pdf"); plt.close(fig); print("wrote fig-scaling.pdf")
show('fig-scaling')

### Figure 10: `fig-threshold-surface`

The audited demographic-parity gap across operating points and selection strengths for one selection experiment.

**Reads:** `accord/selection/threshold_robustness_cells.csv`  
**Drawing code from:** `fig_threshold_surface.py`

In [ ]:
df = pd.read_csv("/Users/rickygole/mimic_local_only/accord_selection/threshold_robustness_cells.csv")
piv = df.pivot_table(index="achieved_rate", columns="beta",
                     values="demographic_parity_gap", aggfunc="mean")
mde = df.pivot_table(index="achieved_rate", columns="beta",
                     values="mde_demographic_parity_gap", aggfunc="mean")
R, B, Z, M = piv.index.values, piv.columns.values, piv.values, mde.values

cmap = LinearSegmentedColormap.from_list("acc", ["#fbfcfd", "#cdd7e0", "#8ba0b6", ACCENT])
fig, ax = plt.subplots(figsize=(6.8, 3.8))
im = ax.pcolormesh(B, R, Z, cmap=cmap, shading="nearest", vmin=0, vmax=0.20)

# veil the underpowered region instead of hatching it
veil = np.where(M > 0.05, 0.62, 0.0)
ax.pcolormesh(B, R, veil, cmap=LinearSegmentedColormap.from_list("v", ["#ffffff00", "#ffffff"]),
              shading="nearest", vmin=0, vmax=1, zorder=2)
ax.contour(B, R, (M > 0.05).astype(float), levels=[0.5],
           colors="0.45", linewidths=0.8, linestyles="dashed", zorder=3)

cs = ax.contour(B, R, Z, levels=[0.05], colors="k", linewidths=1.1, zorder=4)
ax.clabel(cs, fmt={0.05: "0.05"}, fontsize=7, inline=True, inline_spacing=6)

ax.plot(B, [0.1994]*len(B), color="w", lw=1.6, zorder=5)
ax.plot(B, [0.1994]*len(B), color="k", lw=0.9, ls=(0, (5, 2)), zorder=6)
ax.annotate("original E9 operating point", xy=(B[1], 0.1994), xytext=(B[1], 0.132),
            fontsize=7, color="k", zorder=7,
            arrowprops=dict(arrowstyle="-", lw=0.6, color="0.3"))

ax.set_xlabel(r"selection strength through documentation volume  ($\beta$)")
ax.set_ylabel("predicted-positive rate")
ax.set_title("The audit never becomes cleanly blind; it becomes unmeasurable",
             loc="left", fontsize=9.5, pad=16)
ax.text(0, 1.02, "solid line: 0.05 tolerance      shaded pale: minimum detectable gap "
        "exceeds tolerance (underpowered, not passing)",
        transform=ax.transAxes, fontsize=6.9, color="0.4", va="bottom")

cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.043)
cb.set_label("demographic-parity gap", fontsize=8)
cb.ax.tick_params(labelsize=7); cb.outline.set_linewidth(0.5)

out = pathlib.Path(__file__).parent / "fig-threshold-surface.pdf"
fig.savefig(out)
n_under = int((M > 0.05).sum())
print(f"wrote {out.name}  ({Z.shape[0]}x{Z.shape[1]} cells, {n_under} underpowered)")
show('fig-threshold-surface')

### Figure 11: `fig-disc-actionability`

When the system can fire, relative to when the patient leaves.

**Reads:** `schematic, no data file`  
**Drawing code from:** `make_figures.py`, function `make_fig_disc_actionability`

In [ ]:
def make_fig_disc_actionability():
    fig, ax = plt.subplots(figsize=(6.6, 3.4))
    ax.set_xlim(0, 10.5)
    ax.set_ylim(0, 4.0)
    ax.axis("off")

    line_y = 2.3
    events = [
        (1.0, "last nursing note\n(CHARTTIME present)", 3.35),
        (4.0, "discharge order", 3.0),
        (5.4, "DISCHTIME\n(\"patient leaves\")", 3.55),
        (8.2, "discharge summary finalised\n(CHARTDATE only, no CHARTTIME)", 3.05),
    ]
    ax.annotate("", xy=(9.7, line_y), xytext=(0.4, line_y),
                arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=1.2))
    for x, label, ytext in events:
        ax.plot([x], [line_y], marker="o", color=GREY_DARK, markersize=5, zorder=4)
        ax.annotate(label, (x, line_y), xytext=(x, ytext), ha="center", va="bottom", fontsize=6.6,
                    arrowprops=dict(arrowstyle="-", color=GREY_MED, lw=0.6))

    # shaded region: no intervention possible (after DISCHTIME, before trigger)
    ax.axvspan(5.4, 8.2, ymin=(line_y - 0.28) / 4.0, ymax=(line_y + 0.28) / 4.0,
               color=GREY_LIGHT, alpha=0.45, zorder=1)
    ax.text((5.4 + 8.2) / 2, line_y, "no intervention\npossible", ha="center", va="center",
            fontsize=6.3, style="italic", zorder=5)

    ax.plot([8.2], [line_y], marker="X", color=GREY_DARK, markersize=7, zorder=4)
    ax.annotate("system's trigger point\n(discharge summary)", xy=(8.2, line_y - 0.15),
                xytext=(8.2, 1.15), ha="center", va="top", fontsize=6.3,
                arrowprops=dict(arrowstyle="-", color=GREY_MED, lw=0.6))

    ax.annotate("proposed detection point", xy=(4.0, line_y - 0.15), xytext=(2.4, 0.55),
                fontsize=6.3, ha="center", va="top", color=GREY_DARK,
                arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=0.9, linestyle="--"))

    ax.set_title("The actionability flaw: the trigger fires after the patient has left",
                 fontsize=8.5, pad=8)
    fig.tight_layout(rect=[0, 0.10, 1, 1])
    fig.text(0.5, 0.02,
             "Schematic of MIMIC-III field timing (CORRECTIONS.md C6: CHARTTIME is null for every\n"
             "discharge summary). No numeric estimate is shown; not sourced from a results JSON.",
             ha="center", fontsize=5.6, color=GREY_MED)
    fname = "fig-disc-actionability.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:disc-actionability", ["accord/CORRECTIONS.md (C6)"],
             "Structural/schema diagram, not derived from a numeric result file; documents a "
             "MIMIC-III field-availability fact. No CI applies (no estimate is plotted).")


# ============================================================================
# Run all buildable figures + report skipped ones
# ============================================================================
SKIPPED = [
    {
        "label": "fig:asymmetry",
        "chapter": "ch3_data.tex",
        "reason": (
            "Spec requires classified/unclassified mention counts (219,683 / 88,336 / "
            "131,347) and a discharge-vs-nursing note-role split (79,385 / 8,951) for the "
            "heart-failure cohort. No committed file under results/accord contains this "
            "breakdown for the heart-failure cohort: aki/sepsis have phase2 summaries with "
            "total_mentions/classified_mentions, but the primary heart-failure cohort does "
            "not, and no file anywhere gives a discharge-vs-nursing split for any cohort. "
            "mention_count_crosscheck.json gives only a single crosscheck total (218,544 "
            "official vs 227,363 simplified), which is insufficient to reconstruct either "
            "panel. Left as \\figspec."
        ),
    },
    {
        "label": "fig:arch",
        "chapter": "ch6_architecture.tex",
        "reason": (
            "Spec requires the model's parameter budget and per-module status (PPC deleted, "
            "GGHE randn(20,256) bank, 1.54M vs ~7.5M total params, which heads receive "
            "gradient). These facts are documented only in accord/CORRECTIONS.md (C4) as "
            "prose describing source files (accord_phase3_v2_ablations.py) that are not "
            "committed anywhere in this repository -- not in results/accord, not "
            "elsewhere. There is no JSON to point \\includegraphics-adjacent code at. "
            "Left as \\figspec."
        ),
    },
    {
        "label": "fig:falsification",
        "chapter": "ch6_architecture.tex",
        "reason": (
            "Spec requires the authority-parameter init/final values (0.5,1.0) vs "
            "(0.509,1.006) and per-cohort rule-type F1 (HF 0.9603, sepsis 0.0654, AKI "
            "0.0595, held-out 45.2%). None of these numbers appear in any file under "
            "results/accord/. They are documented only as prose in accord/CORRECTIONS.md "
            "(C1, C2), which cites transfer_results_correct.json and "
            "accord_phase3_v2_ablations.py -- neither file exists in this repository. "
            "Left as \\figspec."
        ),
    },
]

make_fig_disc_actionability()
show('fig-disc-actionability')

### Figure 12: `fig-met-exchange`

What each summary statistic trades on this evaluation set: forgone true positives against gained true negatives.

**Reads:** `accord/step3_baselines.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_met_exchange`

In [ ]:
def make_fig_met_exchange():
    baselines, p_baselines = load("diag", "step3_baselines.json")
    systems = {
        "ACCORD": baselines["accord_binary_at_0.5_253core"],
        "GPT-4o zero-shot": baselines["gpt4o_zeroshot"],
        "GPT-4o few-shot": baselines["gpt4o_fewshot"],
        "Llama 3 8B": baselines["llama_baseline"],
    }
    ref = systems["ACCORD"]
    TP0, TN0, FP0, FN0 = ref["tp"], ref["tn"], ref["fp"], ref["fn"]
    P = TP0 + FN0
    N = TN0 + FP0
    mcc0 = mcc_from_counts(TP0, TN0, FP0, FN0)

    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    d = np.linspace(0, P, 200)
    ax.plot(d, d, color=GREY_DARK, linewidth=1.1, label="PABAK (slope 1)")
    slope_ba = N / P
    ax.plot(d, d * slope_ba, color=GREY_MED, linewidth=1.1, linestyle="--",
            label=f"balanced accuracy (slope {slope_ba:.2f})")

    # MCC indifference curve through ACCORD's own operating point, traced
    # numerically: for each delta (forgone TPs), solve for the TN that holds
    # MCC constant at mcc0.
    deltas = np.linspace(0, P, 60)
    dtn_curve = []
    for delta in deltas:
        tp = TP0 - delta
        fn = P - tp

        def f(tn):
            fp = N - tn
            return mcc_from_counts(tp, tn, fp, fn) - mcc0

        try:
            tn_sol = brentq(f, 0, N)
            dtn_curve.append(tn_sol - TN0)
        except ValueError:
            dtn_curve.append(np.nan)
    ax.plot(deltas, dtn_curve, color=GREY_LIGHT, linewidth=2.2, alpha=0.9,
            label=f"MCC indifference (through ACCORD, MCC={mcc0:.3f})")

    markers = {"ACCORD": "o", "GPT-4o zero-shot": "s", "GPT-4o few-shot": "^", "Llama 3 8B": "D"}
    label_offset = {"ACCORD": (8, 10), "GPT-4o zero-shot": (6, 5), "GPT-4o few-shot": (6, 5),
                     "Llama 3 8B": (10, -10)}
    for name, s in systems.items():
        delta = TP0 - s["tp"]
        dtn = s["tn"] - TN0
        ax.scatter([delta], [dtn], marker=markers[name], s=45, color=GREY_DARK,
                   facecolor="white" if name != "ACCORD" else GREY_DARK, zorder=5)
        ax.annotate(name, (delta, dtn), textcoords="offset points", xytext=label_offset[name],
                    fontsize=6.8)

    ax.axhline(0, color=GREY_LIGHT, linewidth=0.6)
    ax.axvline(0, color=GREY_LIGHT, linewidth=0.6)
    ax.set_xlabel(r"$\delta$ = forgone true positives, relative to ACCORD")
    ax.set_ylabel(r"$\Delta$TN, relative to ACCORD")
    ax.set_xlim(-1, P + 1)
    ax.set_ylim(-32, N + 10)
    ax.legend(loc="upper left", fontsize=6.5, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"What each statistic trades (P={P}, N={N}, n={P+N})", fontsize=8.5)
    fig.tight_layout()
    fname = "fig-met-exchange.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:met-exchange", [p_baselines],
             "Indifference lines are closed-form (PABAK, balanced accuracy) or numerically "
             "root-solved (MCC, scipy.optimize.brentq) from the confusion-matrix class counts "
             "in diag/step3_baselines.json; not fabricated curve shapes.")


# ============================================================================
# Figure 9 -- fig:met-sharpened  (ch8_metrics.tex)
# ============================================================================

make_fig_met_exchange()
show('fig-met-exchange')

### Figure 13: `fig-exemplar`

The check that was not run, drawn as a schematic.

**Reads:** `accord/exemplar/exemplar_correctness_recomputed.json`  
**Drawing code from:** `make_figures.py`, function `make_fig_exemplar`

In [ ]:
def make_fig_exemplar():
    exemplar, p_exemplar = load("exemplar", "exemplar_correctness_recomputed.json")
    # The former reference rate was supplied by a task description, never measured
    # (REVISION_LOG R8). Use the measured wrong rate on the resolved candidates.
    _ex = json.load(open(os.path.join(RESULTS, "exemplar", "exemplar_correctness_recomputed.json")))["rule_1_2_pooled"]
    defect_rate = _ex["n_wrong"] / _ex["n_resolved"]

    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")

    # two columns of abstract mention slots (no clinical text)
    col_x = {"Nursing note": 2.2, "Discharge summary": 7.2}
    for label, x in col_x.items():
        ax.text(x, 5.6, label, ha="center", fontsize=8.5, weight="bold")

    nurse_tags = ["M1", "M2", "M3", "M4", "M5"]
    disch_tags = ["CONT", "CONT", "CONT", "CONT", "CESS"]
    for i in range(5):
        y = 4.9 - i * 0.78
        ax.add_patch(mpatches.Rectangle((col_x["Nursing note"] - 0.55, y - 0.28), 1.1, 0.56,
                                          facecolor="white", edgecolor=GREY_DARK))
        ax.text(col_x["Nursing note"], y, nurse_tags[i], ha="center", va="center", fontsize=7.5)
        fc = GREY_FILL if i == 4 else "white"
        ax.add_patch(mpatches.Rectangle((col_x["Discharge summary"] - 0.55, y - 0.28), 1.1, 0.56,
                                          facecolor=fc, edgecolor=GREY_DARK))
        ax.text(col_x["Discharge summary"], y, disch_tags[i], ha="center", va="center", fontsize=7.5)

    # trigger arrow spanning both columns (top)
    ax.annotate("", xy=(col_x["Discharge summary"] - 0.6, 5.15), xytext=(col_x["Nursing note"] + 0.6, 5.15),
                arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=1.1))
    ax.text((col_x["Nursing note"] + col_x["Discharge summary"]) / 2, 5.3,
            "trigger: sets conflict\n(quantifies over the set)", ha="center", fontsize=6.6, color=GREY_MED)

    y0 = 4.9
    y4 = 4.9 - 4 * 0.78
    ax.annotate(".iloc[0]\n(what the model,\nannotator & LLM saw)",
                xy=(col_x["Discharge summary"] - 0.55, y0), xytext=(col_x["Discharge summary"] + 1.5, y0 + 0.5),
                fontsize=6.3, color=GREY_DARK,
                arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=0.9))
    ax.annotate("fix: action-matched\n+ dose-adjacent\n(the CESS mention)",
                xy=(col_x["Discharge summary"] - 0.55, y4), xytext=(col_x["Discharge summary"] + 1.5, y4 - 0.3),
                fontsize=6.3, color=GREY_DARK,
                arrowprops=dict(arrowstyle="-|>", color=GREY_DARK, lw=0.9))

    ax.add_patch(mpatches.FancyBboxPatch((1.2, 0.15), 7.6, 0.75,
                 boxstyle="round,pad=0.05", facecolor=GREY_FILL, edgecolor=GREY_MED))
    ax.text(5.0, 0.52,
            f"For {defect_rate*100:.1f}% of action-conflict candidates, the trigger-set slot and the\n"
            ".iloc[0] slot disagreed (rule_1_2_pooled, exemplar_correctness_recomputed.json)",
            ha="center", va="center", fontsize=7)

    fig.tight_layout()
    fname = "fig-exemplar.pdf"
    fig.savefig(os.path.join(FIGDIR, fname))
    plt.close(fig)
    register(fname, "fig:exemplar", [p_exemplar],
             "Schematic; slots are abstract labels (M1..M5, CONT/CESS action-tag codes), no "
             "clinical text. Only numeric content (41.4%) is sourced.")


# ============================================================================
# Figure 5 -- fig:rule-precision  (ch4_candidates.tex)
#   MODIFIED: the spec's precision-by-rule numbers (pharmacist 2.2/9.7/5.1/3.6
#   and ACCORD 2.3/10.0/5.2/2.4) are not both in any committed file -- only
#   the pharmacist side is (it equals the per-rule positive rate). ACCORD's
#   actual per-rule *precision at its operating threshold* is not released
#   per-rule anywhere in results/accord. What IS released per rule is
#   ACCORD's own AUROC per rule (defect_free_strata.json), which answers the
#   same underlying question ("does the model add anything the rule alone
#   doesn't give you") without inventing a number. Substituted accordingly.
# ============================================================================

make_fig_exemplar()
show('fig-exemplar')

### Save all thirteen back to Drive

In [ ]:
import shutil, pathlib
USED = ["fig-pipeline-status","fig-intro-headline","fig-rule-precision","fig-prevalence-response",
        "fig-falsification","fig-power","fig-forest","fig-disc-strata","fig-scaling",
        "fig-threshold-surface","fig-disc-actionability","fig-met-exchange","fig-exemplar"]
DEST = '/content/drive/MyDrive/This is paper/02_figures'
missing = [n for n in USED if not (OUT/f"{n}.pdf").exists()]
if missing:
    print("NOT saving. These have not been rendered yet:")
    for m in missing: print("  ", m)
else:
    for n in USED: shutil.copy(OUT/f"{n}.pdf", f"{DEST}/{n}.pdf")
    print(f"saved all {len(USED)} figures to Drive")
    print("tell Claude they are updated and the thesis will be rebuilt with them")